# LankaGrid AI Ensemble Learning Assignment

This notebook develops a one-day-ahead Sri Lankan grid generation forecast using a chronological 80/20 split. It compares bagging, boosting, stacking and fixed-weight hybrid ensembles.

In [ ]:
import pandas as pd

source = "data/merged-generation-profile-2017-2026.csv"
data = pd.read_csv(source, na_values=["Data N/A"])
date_col = data.columns[0]
numeric = data.iloc[:, 1:].apply(pd.to_numeric, errors="coerce")
totals = numeric.sum(axis=1).where(numeric.notna().any(axis=1))
daily = pd.DataFrame({"date": pd.to_datetime(data[date_col]), "total_gwh": totals}).dropna()
daily = daily[daily["total_gwh"] > 0].sort_values("date").drop_duplicates("date")
daily.head(), daily.shape

In [ ]:
import sys
import train

raw = train.load_data(source, date_col, "total_gwh")
frame = train.features(raw["energy_gwh"])
split = int(len(frame) * 0.8)
train_df, test_df = frame.iloc[:split], frame.iloc[split:]
X_train, y_train = train_df[train.FEATURES], train_df["energy_gwh"]
X_test, y_test = test_df[train.FEATURES], test_df["energy_gwh"]
len(X_train), len(X_test)

In [ ]:
import joblib
from pathlib import Path

metrics = pd.read_csv("outputs/metrics.csv")
metrics.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}"})

## Interpretation

- The table ranks models using MAE on the chronological held-out period.
- The lowest-MAE model is the strongest saved result, but it is not guaranteed to generalize outside the test period.
- MAE and RMSE are reported in GWh/day; R-squared shows the proportion of held-out variance explained.
- Rolling values are shifted, and lag features use prior observations only, avoiding direct target leakage.
- Stacking and the fixed hybrids are separate demonstrations; hybrid weights were not optimized against the test set.